In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --------------------------------------------------
# ACE Marble Rolling Animation
# Pedagogical visualization of semantic convergence
# --------------------------------------------------

# Energy landscape (2D pedagogical surface)
def energy(x, y):
    valley = 0.35 * (x**2 + 0.7 * y**2)
    ripple = 0.08 * np.sin(2.2 * x) * np.cos(1.8 * y)
    tilt = 0.03 * x
    return valley + ripple + tilt

# Numerical gradient
def grad_energy(x, y, eps=1e-3):
    dEx = (energy(x + eps, y) - energy(x - eps, y)) / (2 * eps)
    dEy = (energy(x, y + eps) - energy(x, y - eps)) / (2 * eps)
    return dEx, dEy

# Simulate marble path with damped gradient descent
def simulate_path(x0, y0, steps=120, lr=0.12, damping=0.88):
    x, y = x0, y0
    vx, vy = 0.0, 0.0
    xs, ys = [x], [y]

    for _ in range(steps - 1):
        gx, gy = grad_energy(x, y)

        vx = damping * vx - lr * gx
        vy = damping * vy - lr * gy

        x += vx
        y += vy

        xs.append(x)
        ys.append(y)

    return np.array(xs), np.array(ys)

# Initial positions: closer = lower O(z), farther = higher O(z)
starts = {
    "precise": (0.9, 0.35),
    "expansive": (2.0, 1.15),
    "speculative": (-1.9, 1.35),
}

# Visual identity
colors = {
    "precise": "#00aa00",
    "expansive": "orange",
    "speculative": "#cc0000",
}

label_offsets = {
    "precise": (0.18, 0.18),
    "expansive": (0.22, -0.28),
    "speculative": (-0.55, 0.30),
}

paths = {name: simulate_path(x0, y0) for name, (x0, y0) in starts.items()}

# Plot domain
x = np.linspace(-3, 3, 220)
y = np.linspace(-2.4, 2.4, 220)
X, Y = np.meshgrid(x, y)
Z = energy(X, Y)

fig, ax = plt.subplots(figsize=(9, 6))
ax.contourf(X, Y, Z, levels=35)
ax.contour(X, Y, Z, levels=14, linewidths=0.5, alpha=0.6)

ax.set_title("ACE Semantic Convergence: Marble Rolling Toward Minimum Energy")
ax.set_xlabel("Semantic drift")
ax.set_ylabel("Contextual deviation")

# Mark approximate attractor
min_idx = np.unravel_index(np.argmin(Z), Z.shape)
x_min = X[min_idx]
y_min = Y[min_idx]
ax.scatter(
    [x_min], [y_min],
    marker="*",
    s=260,
    color="black",
    label="Semantic origin / attractor"
)
ax.legend(loc="upper right")

# Explanatory note
ax.text(
    -2.8, 2.05,
    "Lower O(z) → stronger semantic alignment\nHigher O(z) → semantic drift",
    fontsize=10,
    bbox=dict(facecolor="white", alpha=0.55, edgecolor="none")
)

# Animated artists
lines = {}
points = {}
labels = {}

for name, (xs, ys) in paths.items():
    line, = ax.plot([], [], lw=3.0, color=colors[name])
    sizes = {
    "precise": 8,
    "expansive": 11,
    "speculative": 13,
    }

    point, = ax.plot([], [], marker="o", ms=sizes[name], color=colors[name])
    dx, dy = label_offsets[name]
    text = ax.text(
    xs[0] + dx,
    ys[0] + dy,
    name,
    fontsize=10,
    color="black",
    weight="bold",
    bbox=dict(
        facecolor="white",
        alpha=0.7,
        edgecolor="none",
        boxstyle="round,pad=0.2"
    ))

    lines[name] = line
    points[name] = point
    labels[name] = text

ax.set_xlim(-3, 3)
ax.set_ylim(-2.4, 2.4)

def init():
    for name in paths:
        lines[name].set_data([], [])
        points[name].set_data([], [])
        xs, ys = paths[name]
        dx, dy = label_offsets[name]
        labels[name].set_position((xs[0] + dx, ys[0] + dy))
    return [*lines.values(), *points.values(), *labels.values()]

def update(frame):
    artists = []
    for name, (xs, ys) in paths.items():
        lines[name].set_data(xs[:frame+1], ys[:frame+1])
        points[name].set_data([xs[frame]], [ys[frame]])
        dx, dy = label_offsets[name]
        labels[name].set_position((xs[frame] + dx, ys[frame] + dy))
        artists.extend([lines[name], points[name], labels[name]])
    return artists

anim = FuncAnimation(
    fig,
    update,
    frames=len(next(iter(paths.values()))[0]),
    init_func=init,
    interval=70,
    blit=True
)

plt.close(fig)
HTML(anim.to_jshtml())

from matplotlib.animation import PillowWriter

anim.save("ace_marble_rolling.gif", writer=PillowWriter(fps=14))

print("✅ GIF guardado como ace_marble_rolling.gif")

from google.colab import files
files.download("ace_marble_rolling.gif")

✅ GIF guardado como ace_marble_rolling.gif


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>